# MAFDET gatherer: training on your own PC

Parts 2 and 3 of `gatherer_colab.ipynb` (training and export to GGUF) for a PC with an NVIDIA card, written for
the RTX 5070 Ti (16 GB). The notebook stands alone: it needs neither the MAFDET repository nor git, only Python
and the data generated on Colab. It installs what it needs by itself. Part 1, the data, stays on Colab: vLLM,
which serves the teacher, does not run on Windows. Here everything runs in one session, with no restarts.

**Before you start**

1. Python 3.10 to 3.13 and a recent NVIDIA driver. Best in an environment of its own, in the folder of this
   notebook:
   `python -m venv gatherer-env`, then `gatherer-env\Scripts\pip install notebook` and `gatherer-env\Scripts\jupyter notebook`
   (or open the notebook in VS Code and choose `gatherer-env` as the kernel).
2. Next to this notebook make a `mafdet-gatherer` folder, and copy into it `data.jsonl` from the
   `mafdet-gatherer` folder on Google Drive.
3. To resume a training run started on Colab, copy `checkpoints-Qwen3-0.6B` there too. Without it training
   starts from step 0. Either way, if it stops halfway, running the notebook again resumes from the last checkpoint
   (one every 200 steps).

Disk space: about 5 GB for PyTorch and the other libraries, about 4 GB for the work (base model, two checkpoints,
the merged model, the llama.cpp sources, the GGUF). Only the final GGUF (about 0.6 GB) is needed afterwards.

## 0. Setup

In [ ]:
# PyTorch for CUDA 12.8 or later (the 5070 Ti needs it), then the libraries for training. torch is not imported
# before this cell: if it was already loaded, restart the kernel after an install.
import re
from importlib.metadata import version, PackageNotFoundError
try:
    cuda = re.search(r'\+cu(\d+)', version('torch'))
except PackageNotFoundError:
    cuda = None
if not cuda or int(cuda.group(1)) < 128:
    %pip install -q --no-cache-dir --upgrade torch --index-url https://download.pytorch.org/whl/cu128
# torchao is not needed here, and an old copy makes PEFT refuse to load ("Found an incompatible version of torchao").
import importlib.util
if importlib.util.find_spec('torchao'):
    %pip uninstall -y -q torchao
%pip install -q -U transformers peft accelerate gguf sentencepiece

In [ ]:
import os, sys, subprocess, torch
from pathlib import Path

WORK = str(Path.cwd() / 'mafdet-gatherer')         # next to the notebook: data, checkpoints, models, the GGUF
os.makedirs(WORK, exist_ok=True)
assert torch.cuda.is_available(), 'PyTorch does not see the GPU: update the NVIDIA driver'
major, minor = torch.cuda.get_device_capability(0)
assert f'sm_{major}{minor}' in torch.cuda.get_arch_list(), 'this PyTorch has no kernels for this GPU: install it for CUDA 12.8'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0), '| working folder:', WORK)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv'],
                     capture_output=True, text=True).stdout)

In [ ]:
# The gatherer and its format: the same as in gatherer_colab.ipynb, or a Colab checkpoint will not resume.
# Qwen3-0.6B for 6-8 GB cards (RTX 2060, about 0.9 GB with its cache), Qwen3-1.7B for 16 GB cards: to train
# another one, change STUDENT (its checkpoints go to their own folder).
STUDENT = 'Qwen/Qwen3-0.6B'

GATHER_SEED = ('<think>\nMaterial for the answer, gathered while the user types: for each key term, name or '
               'number already written, one line with what it is and the facts I know about it, or a partial '
               'result.\n\n')
STEP_HEADER = 'New words: "{new}"\n'  # before the lines of each step; the first step names the whole prefix
END = '<|im_end|>'                    # the gatherer ends a step with it: nothing more to add for now

## Part 2: training

In [ ]:
import json, random
from transformers import AutoTokenizer

DATA = f'{WORK}/data.jsonl'
assert os.path.exists(DATA), f'copy data.jsonl from the mafdet-gatherer folder on Google Drive to {WORK}'
tok = AutoTokenizer.from_pretrained(STUDENT)
MAX_LEN = 2048

def examples(rec):
    """One example per step: the prompt MAFDET would build at that point, and the lines to add."""
    out, notes, prev = [], '', ''
    for p, lines in zip(rec['prefixes'], rec['steps']):
        new = p[len(os.path.commonprefix([prev, p])):].strip()
        head = STEP_HEADER.format(new=new[-300:])
        prompt = (tok.apply_chat_template([{'role': 'user', 'content': p}], tokenize=False,
                                          add_generation_prompt=True, enable_thinking=True)
                  + GATHER_SEED + notes + head)
        body = ''.join(l + '\n' for l in lines)
        out.append((prompt, body + END))
        notes += head + body + '\n'
        prev = p
    return out

def encode(prompt, completion):
    p = tok(prompt, add_special_tokens=False).input_ids
    c = tok(completion, add_special_tokens=False).input_ids
    if len(p) + len(c) > MAX_LEN:
        return None
    return {'input_ids': p + c, 'labels': [-100] * len(p) + c}     # loss on the gatherer's lines only

# UTF-8 explicitly: Colab wrote it so, and Windows would read it as cp1252 (other text, other examples).
rows = [json.loads(l) for l in open(DATA, encoding='utf-8')]
rows = [r for r in rows if not r.get('skip')]
random.Random(0).shuffle(rows)
n_eval = max(20, len(rows) // 30)
held_out, train_rows = rows[:n_eval], rows[n_eval:]
train = [e for r in train_rows for e in (encode(*x) for x in examples(r)) if e]
evals = [e for r in held_out for e in (encode(*x) for x in examples(r)) if e]
print(len(train), 'training examples,', len(evals), 'held out')
print(examples(train_rows[0])[1][0][-600:] + '|||' + examples(train_rows[0])[1][1])

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

bf16 = torch.cuda.is_bf16_supported()
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
FULL = bf16 and vram > 70             # big GPU: train every weight; otherwise a LoRA adapter
NAME = STUDENT.split('/')[-1]
print('student:', NAME, '| full fine-tuning' if FULL else '| LoRA', f'| {vram:.0f} GB')
model = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=torch.bfloat16 if bf16 else torch.float32)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
if not FULL:
    model = get_peft_model(model, LoraConfig(
        r=64, lora_alpha=128, lora_dropout=0.05, task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    model.print_trainable_parameters()

def collate(batch):
    n = max(len(b['input_ids']) for b in batch)
    pad = tok.pad_token_id
    return {'input_ids': torch.tensor([b['input_ids'] + [pad] * (n - len(b['input_ids'])) for b in batch]),
            'labels': torch.tensor([b['labels'] + [-100] * (n - len(b['labels'])) for b in batch]),
            'attention_mask': torch.tensor([[1] * len(b['input_ids']) + [0] * (n - len(b['input_ids'])) for b in batch])}

# 16 examples per update. On a 16 GB card, 4 at a time: the logits over the ~150k-word vocabulary of 8 long
# examples alone take about 10 GB.
import math
batch = 8 if vram > 30 else 4
steps = math.ceil(len(train) / 16) * 3
args = TrainingArguments(
    output_dir=f'{WORK}/checkpoints-{NAME}', num_train_epochs=3, per_device_train_batch_size=batch,
    gradient_accumulation_steps=16 // batch, learning_rate=1e-5 if FULL else 2e-4, lr_scheduler_type='cosine',
    warmup_steps=max(1, steps * 3 // 100), weight_decay=0.0 if FULL else 0.01,
    logging_steps=20, eval_strategy='steps', eval_steps=200, save_steps=200, save_total_limit=2,
    bf16=bf16, fp16=not bf16, report_to='none', remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=evals, data_collator=collate)

# Resume from the last checkpoint, from Colab or from an earlier run here. One cut off while it was being
# written lacks some of these files: delete it and the one before is used.
last = get_last_checkpoint(args.output_dir) if os.path.isdir(args.output_dir) else None
if last:
    weights = 'model.safetensors' if FULL else 'adapter_model.safetensors'
    missing = [f for f in (weights, 'optimizer.pt', 'scheduler.pt', 'trainer_state.json')
               if not os.path.exists(f'{last}/{f}')]
    assert not missing, f'{last} is incomplete (missing {missing}): delete it and run this cell again'
print('resuming from', last, f'of {steps} steps' if last else f'| starting from step 0 of {steps}')
trainer.train(resume_from_checkpoint=last)

In [ ]:
# Merge the adapter into the model (full fine-tuning has none) and save it.
merged = model if FULL else model.merge_and_unload()
merged.save_pretrained(f'{WORK}/merged-{NAME}', safe_serialization=True)
tok.save_pretrained(f'{WORK}/merged-{NAME}')
print('saved to', f'{WORK}/merged-{NAME}')

In [ ]:
# Try it on held-out messages: does it gather material, and does it stay on the words typed so far?
merged.eval()
for r in held_out[:4]:
    print('=' * 80)
    for prompt, gold in examples(r):
        ids = tok(prompt, return_tensors='pt', add_special_tokens=False).input_ids.to(merged.device)
        with torch.no_grad():
            out = merged.generate(ids, max_new_tokens=160, do_sample=False,
                                  eos_token_id=tok.convert_tokens_to_ids(END))
        print('>>', prompt[prompt.rindex('New words'):].strip())
        print(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip() or '   (nothing)')

## Part 3: export to GGUF

In [ ]:
# The converter comes with the llama.cpp sources: downloaded as a zip from GitHub, no git needed.
import io, urllib.request, zipfile
LLAMA = f'{WORK}/llama.cpp-master'
if not os.path.isdir(LLAMA):
    url = 'https://github.com/ggml-org/llama.cpp/archive/refs/heads/master.zip'
    with urllib.request.urlopen(url) as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(WORK)
name = STUDENT.split('/')[-1]
GGUF = f'{WORK}/mafdet-gatherer-{name}-Q8_0.gguf'     # MAFDET finds the tokenizer from this name: keep it
r = subprocess.run([sys.executable, f'{LLAMA}/convert_hf_to_gguf.py', f'{WORK}/merged-{name}',
                    '--outtype', 'q8_0', '--outfile', GGUF], capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-3000:])
r.check_returncode()
print('written', GGUF, f'({os.path.getsize(GGUF) / 2**30:.2f} GB)')

Copy `mafdet-gatherer-Qwen3-0.6B-Q8_0.gguf` from the `mafdet-gatherer` folder into the MAFDET project folder on the
PC that runs it, without renaming it: from its name MAFDET finds the tokenizer (Qwen/Qwen3-0.6B). There it is
the gatherer next to the large model: `--gatherer mafdet-gatherer-Qwen3-0.6B-Q8_0.gguf`.
Once it works, the rest of `mafdet-gatherer` (checkpoints, merged model, llama.cpp) can be deleted.